In [4]:
import pandas as pd
import numpy as np
import dai

def main(datasources, start_date, end_date):
    bar1m_table = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")  # 高频, 仅盘口因子用
    financial_table = datasources.get("financial", "bigalpha_2026_financial")
    bar1d_table = "bigalpha_2026_bar1d"                   # 日频, 优先使用
    factorlib_table = "bigalpha_2026_factorlib"           # 日频预计算
    exposure_table = "bigalpha_2026_exposure"             # 日频暴露

    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)
    calc_start = start_dt - pd.Timedelta(days=360)  # 最大回看20个交易日，360天足够

    # 读取close和net_profit_rate_ttm
    df = dai.query(f"SELECT date, instrument, close, net_profit_rate_ttm FROM {factorlib_table}", filters={'date': [str(calc_start), str(end_date)]}).df()
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])
    assert all(c in df.columns for c in ['date', 'instrument']), "缺少必需列 date 或 instrument"
    df['date'] = pd.to_datetime(df['date'])
    df = df.drop_duplicates(['date', 'instrument'])

    # 数据清洗：数值列转换、前向填充、缺失填0
    for col in ['close', 'net_profit_rate_ttm']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df.groupby('instrument')[col].ffill()
        df[col] = df[col].fillna(0)

    # 按instrument和date排序
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)

    # 计算ts_rank(close, 5) - 5日内的close排名
    df['ts_rank_close_5'] = df.groupby('instrument')['close'].transform(lambda x: x.rolling(5, min_periods=5).rank(pct=True))

    # 计算ts_rank(net_profit_rate_ttm, 12) - 12期内的net_profit_rate_ttm排名
    df['ts_rank_npr_12'] = df.groupby('instrument')['net_profit_rate_ttm'].transform(lambda x: x.rolling(12, min_periods=12).rank(pct=True))

    # 计算ts_rank(close, 20) - 20日内的close排名
    df['ts_rank_close_20'] = df.groupby('instrument')['close'].transform(lambda x: x.rolling(20, min_periods=20).rank(pct=True))

    # 计算rank(ts_rank(close, 5)) - 截面排名
    df['rank_ts_rank_close_5'] = df.groupby('date')['ts_rank_close_5'].rank(pct=True)

    # 计算rank(ts_rank(net_profit_rate_ttm, 12)) - 截面排名
    df['rank_ts_rank_npr_12'] = df.groupby('date')['ts_rank_npr_12'].rank(pct=True)

    # 计算rank(ts_rank(close, 20)) - 截面排名
    df['rank_ts_rank_close_20'] = df.groupby('date')['ts_rank_close_20'].rank(pct=True)

    # 计算因子值: rank(ts_rank(close,5)) * rank(ts_rank(net_profit_rate_ttm,12)) * (-1 * rank(ts_rank(close,20)))
    try:
        df['factor'] = df['rank_ts_rank_close_5'] * df['rank_ts_rank_npr_12'] * (-1 * df['rank_ts_rank_close_20'])
    except Exception:
        df['factor'] = np.nan
    df['factor'] = df['factor'].replace([np.inf, -np.inf], np.nan)

    result = df[['date', 'instrument', 'factor']]
    result['factor'] = result['factor'].fillna(0)
    if not result.empty:
        result = result.dropna(subset=['factor'])
    result = result[result['date'] >= str(start_dt)]
    return result

[2026-07-16 14:59:51] [info     ] 本地测试，区间：2024-01-01 ~ 2024-12-31

=== 数据覆盖检查 ===
返回数据形状: (236584, 3)
日期范围: 2024-01-04 ~ 2024-12-31
因子缺失值数量: 0
每日平均股票数: 986
[2026-07-16 15:00:01] [info     ] 本地测试，区间：2024-01-01 ~ 2024-12-31

=== 数据覆盖检查 ===
返回数据形状: (236584, 3)
日期范围: 2024-01-04 ~ 2024-12-31
因子缺失值数量: 0
每日平均股票数: 986

实际存在的交易日数量: 240
首日: 2024-01-04
末日: 2024-12-31
⚠️ 存在较大间隔: 2024-02-08 -> 2024-02-19 (间隔11天)
⚠️ 存在较大间隔: 2024-04-03 -> 2024-04-08 (间隔5天)
⚠️ 存在较大间隔: 2024-04-30 -> 2024-05-06 (间隔6天)
⚠️ 存在较大间隔: 2024-06-07 -> 2024-06-11 (间隔4天)
⚠️ 存在较大间隔: 2024-09-13 -> 2024-09-18 (间隔5天)
⚠️ 存在较大间隔: 2024-09-30 -> 2024-10-08 (间隔8天)

✅ 每天因子值均有有效值（非全 NaN）。

前5行数据:
         date instrument    factor
0  2024-01-04  000006.SZ -0.008095
1  2024-01-05  000006.SZ -0.047213
2  2024-01-08  000006.SZ -0.093443
3  2024-01-09  000006.SZ -0.071124
4  2024-01-10  000006.SZ -0.047213

后5行数据:
              date instrument    factor
236579  2024-12-09  688800.SH -0.218390
236580  2024-12-10  688800.SH -0.098512
236581  2024-